In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX

np.random.seed(42)

# --------------------
# Paths
# --------------------
SPLIT_DIR  = Path("../backend/data/splits_70_15_15")
TRAIN_PATH = SPLIT_DIR / "train.parquet"
VAL_PATH   = SPLIT_DIR / "val.parquet"
TEST_PATH  = SPLIT_DIR / "test.parquet"

EVAL_HIVES_MASTER = Path("../backend/data/model_comparison_fair/eval_hives_master.csv")

OUT_DIR = Path("../backend/data/arima_outputs_fair")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_TAG = "arima_fair"

# --------------------
# Settings
# --------------------
OBS_COLS = ["temperature", "humidity", "audio_density"]
D = len(OBS_COLS)

MIN_ROWS_VAL  = 200
MIN_ROWS_TEST = 200

# Small grid (SARIMAX is slow)
P_SET = [0, 1, 2]
D_SET = [0, 1]
Q_SET = [0, 1, 2]

MIN_HIVES_FOR_ORDER = 10
MIN_SERIES_LEN_TO_FIT = 50

FREQ = "15min"  # fixed time grid (no imputation)
TREND = "c"
ENFORCE_STATIONARITY = False
ENFORCE_INVERTIBILITY = False

# --------------------
# Helpers
# --------------------
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["tag_number"] = pd.to_numeric(df["tag_number"], errors="coerce").astype("Int64")
    for c in OBS_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["published_at", "tag_number"])
    return df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)

def usable_rows(df_h: pd.DataFrame) -> int:
    z = df_h[OBS_COLS].to_numpy(float)
    return int(np.isfinite(z).any(axis=1).sum())

def train_std_from_train(train_df: pd.DataFrame) -> np.ndarray:
    std = train_df[OBS_COLS].astype(float).std(skipna=True).to_numpy()
    std = np.where(np.isfinite(std) & (std > 1e-12), std, 1.0)
    return std

def rmse_per_dim(y: np.ndarray, yhat: np.ndarray) -> np.ndarray:
    y = np.asarray(y, float)
    yhat = np.asarray(yhat, float)
    out = np.full((y.shape[1],), np.nan, float)
    for d in range(y.shape[1]):
        m = np.isfinite(y[:, d]) & np.isfinite(yhat[:, d])
        if m.sum() > 0:
            out[d] = float(np.sqrt(np.mean((y[m, d] - yhat[m, d]) ** 2)))
    return out

def agg_zrmse(rmse_raw: np.ndarray, train_std: np.ndarray) -> float:
    rmse_raw = np.asarray(rmse_raw, float)
    m = np.isfinite(rmse_raw) & np.isfinite(train_std) & (train_std > 1e-12)
    if m.sum() == 0:
        return np.nan
    return float(np.mean(rmse_raw[m] / train_std[m]))

def persistence_baseline_missing_safe(z: np.ndarray) -> np.ndarray:
    T, d = z.shape
    out = np.full((T, d), np.nan, float)
    last = np.full((d,), np.nan, float)
    for t in range(T):
        out[t] = last
        m = np.isfinite(z[t])
        last[m] = z[t, m]
    return out

def load_eval_hives(path: Path) -> list[int]:
    df = pd.read_csv(path)
    col = "hive_id" if "hive_id" in df.columns else df.columns[0]
    return sorted(pd.to_numeric(df[col], errors="coerce").dropna().astype(int).tolist())

def build_grid(train_ts: pd.Series, split_ts: pd.Series) -> pd.DatetimeIndex:
    tmin = min(pd.to_datetime(train_ts, utc=True).min(), pd.to_datetime(split_ts, utc=True).min())
    tmax = max(pd.to_datetime(train_ts, utc=True).max(), pd.to_datetime(split_ts, utc=True).max())
    return pd.date_range(start=tmin, end=tmax, freq=FREQ, tz="UTC")

def series_on_grid(df_h: pd.DataFrame, col: str, grid: pd.DatetimeIndex) -> pd.Series:
    s = df_h.set_index("published_at")[col].sort_index()
    return s.reindex(grid)  # keep NaNs, no imputation

def fit_sarimax_train_only(s_train: pd.Series, order: tuple[int,int,int]):
    if int(s_train.notna().sum()) < MIN_SERIES_LEN_TO_FIT:
        return None
    try:
        model = SARIMAX(
            s_train,
            order=order,
            trend=TREND,
            enforce_stationarity=ENFORCE_STATIONARITY,
            enforce_invertibility=ENFORCE_INVERTIBILITY,
        )
        return model.fit(disp=False)
    except Exception:
        return None

def fair_one_step_forecast_stream(res_train, s_future: pd.Series) -> np.ndarray:
    """
    One-step-ahead forecasts on future segment with TRAIN-fitted parameters.
    predicted at time t using information up to t-1.
    """
    ext = res_train.extend(s_future, refit=False)
    pred = ext.get_prediction(
        start=0,
        end=len(s_future) - 1,
        information_set="predicted"
    ).predicted_mean
    return np.asarray(pred, float)

# --------------------
# Load splits
# --------------------
print("Loading splits...")
train_df = clean_df(pd.read_parquet(TRAIN_PATH))
val_df   = clean_df(pd.read_parquet(VAL_PATH))
test_df  = clean_df(pd.read_parquet(TEST_PATH))

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("Hives (train/val/test):",
      train_df["tag_number"].nunique(),
      val_df["tag_number"].nunique(),
      test_df["tag_number"].nunique())

TRAIN_STD = train_std_from_train(train_df)
print("TRAIN std:", dict(zip(OBS_COLS, TRAIN_STD)))

# --------------------
# FAIR eval hives (VAL ∩ TEST)
# --------------------
if EVAL_HIVES_MASTER.exists():
    eval_hives = load_eval_hives(EVAL_HIVES_MASTER)
    print(f"Loaded eval_hives_master.csv: {len(eval_hives)} hives")
else:
    usable_val = [int(h) for h in val_df["tag_number"].dropna().unique()
                  if usable_rows(val_df[val_df["tag_number"] == h]) >= MIN_ROWS_VAL]
    usable_test = [int(h) for h in test_df["tag_number"].dropna().unique()
                   if usable_rows(test_df[test_df["tag_number"] == h]) >= MIN_ROWS_TEST]
    eval_hives = sorted(list(set(usable_val) & set(usable_test)))
    print(f"Built FAIR eval_hives (VAL∩TEST): {len(eval_hives)} hives")

if len(eval_hives) == 0:
    raise ValueError("No FAIR eval_hives found. Lower MIN_ROWS_* or check splits.")

pd.DataFrame({"hive_id": eval_hives}).to_csv(OUT_DIR / f"{MODEL_TAG}_eval_hives.csv", index=False)

# --------------------
# Tune ARIMA order on VAL (TRAIN-fit, one-step-ahead)
# --------------------
orders = [(p, d, q) for p in P_SET for d in D_SET for q in Q_SET]
tuning_rows = []

print("\nTuning ARIMA order on VAL (TRAIN-fit, one-step-ahead)...")
for order in orders:
    scores = []
    aics = []
    bics = []
    used_hives = 0

    for hive_id in eval_hives:
        tr = train_df[train_df["tag_number"].astype(int) == hive_id].sort_values("published_at")
        va = val_df[val_df["tag_number"].astype(int) == hive_id].sort_values("published_at")
        if tr.empty or va.empty:
            continue

        grid = build_grid(tr["published_at"], va["published_at"])
        last_train_t = pd.to_datetime(tr["published_at"], utc=True).max()
        future_mask = grid > last_train_t

        preds_val = np.full((len(va), D), np.nan, float)
        ok = True
        per_var_aic = []
        per_var_bic = []

        for j, col in enumerate(OBS_COLS):
            s_train_full = series_on_grid(tr, col, grid)
            s_val_full   = series_on_grid(va, col, grid)

            s_train = s_train_full[~future_mask]
            s_future = s_val_full[future_mask]

            res = fit_sarimax_train_only(s_train, order=order)
            if res is None:
                ok = False
                break

            per_var_aic.append(float(getattr(res, "aic", np.nan)))
            per_var_bic.append(float(getattr(res, "bic", np.nan)))

            yhat_future = fair_one_step_forecast_stream(res, s_future)
            yhat_series = pd.Series(yhat_future, index=s_future.index)

            preds_val[:, j] = yhat_series.reindex(
                pd.to_datetime(va["published_at"], utc=True)
            ).to_numpy(float)

        if not ok:
            continue

        z_val = va[OBS_COLS].to_numpy(float)
        rmse = rmse_per_dim(z_val, preds_val)
        scores.append(agg_zrmse(rmse, TRAIN_STD))
        aics.append(float(np.nanmean(per_var_aic)))
        bics.append(float(np.nanmean(per_var_bic)))
        used_hives += 1

    tuning_rows.append({
        "order_p": order[0],
        "order_d": order[1],
        "order_q": order[2],
        "median_val_agg_zrmse": float(np.nanmedian(scores)) if len(scores) else np.nan,
        "mean_val_agg_zrmse": float(np.nanmean(scores)) if len(scores) else np.nan,
        "mean_val_aic": float(np.nanmean(aics)) if len(aics) else np.nan,
        "mean_val_bic": float(np.nanmean(bics)) if len(bics) else np.nan,
        "num_hives_used": int(used_hives),
    })

tuning_df = pd.DataFrame(tuning_rows)

# discard unreliable orders
mask_few = tuning_df["num_hives_used"] < MIN_HIVES_FOR_ORDER
tuning_df.loc[mask_few, ["median_val_agg_zrmse", "mean_val_agg_zrmse"]] = np.nan

tuning_df = tuning_df.sort_values(["median_val_agg_zrmse", "mean_val_agg_zrmse"]).reset_index(drop=True)
if tuning_df["median_val_agg_zrmse"].notna().sum() == 0:
    raise ValueError("All ARIMA orders failed or used too few hives. Relax constraints.")

best = tuning_df.iloc[0]
best_order = (int(best["order_p"]), int(best["order_d"]), int(best["order_q"]))

tuning_path = OUT_DIR / f"{MODEL_TAG}_order_tuning_results.csv"
tuning_df.to_csv(tuning_path, index=False)

print("\nBest ARIMA order on VAL:", best_order)
print("Saved:", tuning_path)

# --------------------
# Final FAIR evaluation on VAL + TEST (must succeed for ALL 3 vars)
# --------------------
print("\nRunning final ARIMA FAIR on VAL + TEST...")

step_rows = []
summary_rows = []

for split_name, df_split in [("val", val_df), ("test", test_df)]:
    for hive_id in eval_hives:
        tr = train_df[train_df["tag_number"].astype(int) == hive_id].sort_values("published_at")
        sp = df_split[df_split["tag_number"].astype(int) == hive_id].sort_values("published_at")
        if tr.empty or sp.empty:
            continue

        grid = build_grid(tr["published_at"], sp["published_at"])
        last_train_t = pd.to_datetime(tr["published_at"], utc=True).max()
        future_mask = grid > last_train_t

        preds = np.full((len(sp), D), np.nan, float)

        ok = True
        for j, col in enumerate(OBS_COLS):
            s_train_full = series_on_grid(tr, col, grid)
            s_split_full = series_on_grid(sp, col, grid)

            s_train = s_train_full[~future_mask]
            s_future = s_split_full[future_mask]

            res = fit_sarimax_train_only(s_train, order=best_order)
            if res is None:
                ok = False
                break

            yhat_future = fair_one_step_forecast_stream(res, s_future)
            yhat_series = pd.Series(yhat_future, index=s_future.index)

            preds[:, j] = yhat_series.reindex(
                pd.to_datetime(sp["published_at"], utc=True)
            ).to_numpy(float)

        # FAIRNESS FIX: require all 3 variables
        if not ok:
            continue

        z = sp[OBS_COLS].to_numpy(float)

        step_rows.append(pd.DataFrame({
            "published_at": pd.to_datetime(sp["published_at"].to_numpy(), utc=True),
            "hive_id": int(hive_id),
            "split": split_name,
            "y_true_temperature": z[:, 0],
            "y_true_humidity": z[:, 1],
            "y_true_audio_density": z[:, 2],
            "y_pred_temperature": preds[:, 0],
            "y_pred_humidity": preds[:, 1],
            "y_pred_audio_density": preds[:, 2],
        }))

        base = persistence_baseline_missing_safe(z)
        rmse_base = rmse_per_dim(z, base)
        rmse_arima = rmse_per_dim(z, preds)

        base_agg = agg_zrmse(rmse_base, TRAIN_STD)
        arima_agg = agg_zrmse(rmse_arima, TRAIN_STD)

        impr = np.nan
        if np.isfinite(base_agg) and base_agg > 0 and np.isfinite(arima_agg):
            impr = 100.0 * (base_agg - arima_agg) / base_agg

        summary_rows.append({
            "hive_id": int(hive_id),
            "split": split_name,
            "baseline_agg_zrmse": base_agg,
            "arima_agg_zrmse": arima_agg,
            "improvement_pct": impr,
            "baseline_rmse_temp": rmse_base[0],
            "baseline_rmse_humid": rmse_base[1],
            "baseline_rmse_audio": rmse_base[2],
            "arima_rmse_temp": rmse_arima[0],
            "arima_rmse_humid": rmse_arima[1],
            "arima_rmse_audio": rmse_arima[2],
            "n_rows": int(len(sp)),
        })

out_df = pd.concat(step_rows, ignore_index=True) if step_rows else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows)

if out_df.empty or summary_df.empty:
    raise RuntimeError("No ARIMA outputs produced. Check fit constraints and data coverage.")

# Save outputs
parquet_all  = OUT_DIR / f"{MODEL_TAG}_val_test_forecast.parquet"
parquet_val  = OUT_DIR / f"{MODEL_TAG}_forecasts_val.parquet"
parquet_test = OUT_DIR / f"{MODEL_TAG}_forecasts_test.parquet"

out_df.to_parquet(parquet_all, index=False)
out_df[out_df["split"] == "val"].to_parquet(parquet_val, index=False)
out_df[out_df["split"] == "test"].to_parquet(parquet_test, index=False)

summary_path = OUT_DIR / f"{MODEL_TAG}_summary_metrics_per_hive.csv"
summary_df.to_csv(summary_path, index=False)

run_config_path = OUT_DIR / f"{MODEL_TAG}_run_config.csv"
pd.DataFrame([{
    "MODEL_TAG": MODEL_TAG,
    "best_order_p": best_order[0],
    "best_order_d": best_order[1],
    "best_order_q": best_order[2],
    "FREQ": FREQ,
    "TREND": TREND,
    "ENFORCE_STATIONARITY": ENFORCE_STATIONARITY,
    "ENFORCE_INVERTIBILITY": ENFORCE_INVERTIBILITY,
    "MIN_ROWS_VAL": MIN_ROWS_VAL,
    "MIN_ROWS_TEST": MIN_ROWS_TEST,
    "MIN_SERIES_LEN_TO_FIT": MIN_SERIES_LEN_TO_FIT,
    "MIN_HIVES_FOR_ORDER": MIN_HIVES_FOR_ORDER,
    "n_eval_hives_requested": int(len(eval_hives)),
    "n_eval_hives_succeeded_val": int(summary_df[summary_df["split"]=="val"]["hive_id"].nunique()),
    "n_eval_hives_succeeded_test": int(summary_df[summary_df["split"]=="test"]["hive_id"].nunique()),
    "TRAIN_STD": ",".join([f"{x:.6g}" for x in TRAIN_STD]),
}]).to_csv(run_config_path, index=False)

print("\nSaved ARIMA FAIR outputs in:", OUT_DIR)
print(" -", OUT_DIR / f"{MODEL_TAG}_eval_hives.csv")
print(" -", tuning_path)
print(" -", parquet_all)
print(" -", parquet_val)
print(" -", parquet_test)
print(" -", summary_path)
print(" -", run_config_path)

print("\nDONE — ARIMA FAIR evaluation finished.")


Loading splits...
Train: (560733, 31) Val: (120160, 31) Test: (120187, 31)
Hives (train/val/test): 52 52 52
TRAIN std: {'temperature': np.float64(2.0893432572964894), 'humidity': np.float64(6.593098136635135), 'audio_density': np.float64(7.089253512944294)}
Loaded eval_hives_master.csv: 26 hives

Tuning ARIMA order on VAL (TRAIN-fit, one-step-ahead)...


C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['refit']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['refit']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['refit']).Passing


Best ARIMA order on VAL: (1, 1, 2)
Saved: ..\backend\data\arima_outputs_fair\arima_fair_order_tuning_results.csv

Running final ARIMA FAIR on VAL + TEST...


C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['refit']).Passing unknown keyword arguments will raise a TypeError beginning in version 0.15.
  warnings.warn(msg, FutureWarning)
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\tsa\statespace\representation.py:374: FutureWarning: Unknown keyword arguments: dict_keys(['refit']).Passing unknown keyword arguments will raise a TypeError b


Saved ARIMA FAIR outputs in: ..\backend\data\arima_outputs_fair
 - ..\backend\data\arima_outputs_fair\arima_fair_eval_hives.csv
 - ..\backend\data\arima_outputs_fair\arima_fair_order_tuning_results.csv
 - ..\backend\data\arima_outputs_fair\arima_fair_val_test_forecast.parquet
 - ..\backend\data\arima_outputs_fair\arima_fair_forecasts_val.parquet
 - ..\backend\data\arima_outputs_fair\arima_fair_forecasts_test.parquet
 - ..\backend\data\arima_outputs_fair\arima_fair_summary_metrics_per_hive.csv
 - ..\backend\data\arima_outputs_fair\arima_fair_run_config.csv

DONE — ARIMA FAIR evaluation finished.
